In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [2]:
import sys
sys.path.append('../')
from model.lrp import LRP
from model.mmst_lrp import MMST_ViT_LRP

import Inference as inf
from src import dataloader
from model import configs, engine

In [3]:
import os
seed = 1987 
import torch  
import numpy as np
torch.manual_seed(seed)
np.random.seed(seed)
from IPython.core.display import HTML, display
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import tiktoken  
from typing import Union, Dict, List


### Text

In [22]:
class TextAttenVis(nn.Module):
    def __init__(self, tokens: List[int], attns_arr, layer: int, head: int):
        super().__init__()
        self.tokens = tokens
        self.attns_arr = attns_arr  # Assuming attns_arr is a tensor with shape [H, X, Y, L]
        self.layer = layer
        self.head = head
        self.TextEncoder = tiktoken.get_encoding('gpt2')
        
    def decode(self):
        """Decode tokens using the TextEncoder."""
        # Decode both the text and get token offsets
        decoded_text = self.TextEncoder.decode(self.tokens)
        token_bytes = [self.TextEncoder.decode_single_token_bytes(token) for token in self.tokens]
        words = [t.decode('utf-8') for t in token_bytes]
        return words
    
    def calc_attn(self):
        """Calculate the average attention score for a specific layer and head, excluding the first token."""
        # Extract the attention scores for the specified head and layer
        attn_scores = self.attns_arr[self.head, :, :, self.layer]
        # Exclude the first token's attention scores (position 0)
        attn_scores = attn_scores[1:, 1:]
        # Calculate the average across columns
        avg_attn_scores = np.mean(attn_scores, axis=0)

        norm_attn_scores = self.normalize_array(avg_attn_scores)
        return norm_attn_scores
    
    def plot(self):
        """Use decoded text and attention scores to create a visualization."""
        words = self.decode()
        importances = self.calc_attn()

        self.visualize_text(words, importances)

    def normalize_array(self, values):
        min_old = values.min()
        max_old = values.max()
        min_new, max_new=0, 1
        normalized_values = [(value - min_old) / (max_old - min_old) * (max_new - min_new) + min_new for value in values]
        return np.array(normalized_values, dtype= np.float32)
    
    def format_special_tokens(self, word):
        # Strip underscores often used in tokenized outputs
        return word.replace('_', ' ')


    def _get_color(self, attr):
        # clip values to prevent CSS errors (Values should be from [-1,1])
        attr = max(-1, min(1, attr))
        if attr > 0:
            hue = 120
            sat = 75
            lig = 100 - int(50 * attr)
        else:
            hue = 0
            sat = 75
            lig = 100 - int(-40 * attr)
        return "hsl({}, {}%, {}%)".format(hue, sat, lig)

    def format_word_importances(self, words, importances):
        tags = ["<td>"]
        for word, importance in zip(words, importances):
            color = self._get_color(importance)
            tags.append(
                '<mark style="background-color: {color}; opacity:1.0; line-height:1.75">'
                '<font color="black"> {word} </font></mark>'.format(color=color, word=word)
            )
        tags.append("</td>")
        return "".join(tags)

    def visualize_text(self, words, importances, legend=True):
        assert len(words) == len(importances), "Words and importances must have the same length."
        
        dom = ["<table style='width: 100%;'>"]
        dom.append(
            "<tr>{}</tr>".format(self.format_word_importances(words, importances))
        )
        
        if legend:
            dom.append(
                '<div style="border-top: 1px solid; margin-top: 5px; padding-top: 5px; display: inline-block">'
            )
            dom.append("<b>Legend: </b>")
            for value, label in zip([-1, 0, 1], ["Negative", "Neutral", "Positive"]):
                dom.append(
                    '<span style="display: inline-block; width: 20px; height: 10px; border: 1px solid; background-color: {value};"></span> {label}  '.format(
                        value=self._get_color(value), label=label
                    )
                )
            dom.append("</div>")
        
        dom.append("</table>")
        html = HTML("".join(dom))
        display(html)
        return html


In [64]:
text_attns = np.load('/home/hkaman/Documents/multimodel-transformers-vye/attn_scores.npy', allow_pickle=True).item()
idx = 10
print(len(text_attns[idx]['tokens']), text_attns[idx]['text_array'].shape)

248 (8, 249, 249, 6)


In [23]:
_ = TextAttenVis(text_attns[10]['tokens'], text_attns[10]['text_array'], layer = -1, head= -1).plot()

### LRP

In [4]:
config = configs.Configs(
    img_size = 16, 
    patch_size = 8, 
    embed_dim = 768, 
    mlp_dim = 512, 
    pool = 'cls',
    in_channels = 8,
    out_channels = 1, 
    num_heads = 8, 
    num_layers = 6, 
    cond = False,
    multi_conv = False,
    attn_dropout = 0.3, 
    proj_dropout = 0.3, 
    drop_path = 0.0,
    post_norm = False, 
    vis = True, 
    tokenizer = 'EC',
    mask_modality = None
    ).call()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MMST_ViT_LRP(config, cond = False).to(device)

exp = '08_MMST_LRP_Test'
exp_output_dir = '/data2/hkaman/Projects/ViT/EXPs/' + 'EXP_' + exp
best_model_name = os.path.join(exp_output_dir, 'best_model_' + exp + '.pth')

model.load_state_dict(torch.load(best_model_name))

<All keys matched successfully>

In [6]:
model.eval()
attribution_generator = LRP(model)

In [7]:
data_loader_training, data_loader_validate, data_loader_test = dataloader.dataloaders(
    batch_size = 1, 
    img_size = 16,
    in_channels = 8, 
    resmapling_status = False,
    data = 's2',
    exp_name = 'test'
    )

(13875, 36) | (8233, 36) | (12443, 36)


In [8]:
data_dict_stt = {}
for batch, sample in enumerate(data_loader_training):
    data_dict_stt[sample['block'][0]] = {
    'image': sample['image'][0].to(device),
    'met':sample['met'][0].to(device),
    'text': sample['EmbText'],
    'yz': sample['YZ'][0],
    'mask':sample['mask'][0]
}

In [ ]:
for block, data in data_dict_stt.items():
    # print(data['image'].unsqueeze(0).shape, data['met'].unsqueeze(0).shape, data['yz'].unsqueeze(0).shape)
    imgmet_attr, context_attr = attribution_generator.generate_LRP(data['image'].unsqueeze(0), 
                                                                 data['text'], 
                                                                 data['met'].unsqueeze(0), 
                                                                 data['yz'].unsqueeze(0),)
    
    print(imgmet_attr[0].shape, imgmet_attr[1].shape, context_attr.shape)
    # transformer_attribution = transformer_attribution.reshape(1, 4, 4, 4)
    # transformer_attribution4 = (transformer_attribution - transformer_attribution.min()) / (transformer_attribution.max() - transformer_attribution.min())
    # data_dict_stt[date_time]['lpr'] = transformer_attribution4.data.cpu().numpy()